In [1]:
import os
import random
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split

# ------------------------------
# Define Paths and Constants
# ------------------------------
image_dirs = [
    r'C:\profolders\Collage stuff\Sem 4 project\DL\Datasets\smote_maximum_clarity'
]
mask_dir = r'C:\profolders\Collage stuff\Sem 4 project\DL\Datasets\smote_maximum_clarity_masks'
metadata_path = r'C:\profolders\Collage stuff\Sem 4 project\DL\Datasets\smote_maximum_clarity\smote_metadata.csv'
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 16

# ------------------------------
# Synchronized Transformations
# ------------------------------
class SynchronizedTransform:
    """
    Applies the same random flips and rotations to both image and mask.
    """
    def __init__(self, flip_prob=0.5, rotation_degrees=(0, 90, 180, 270)):
        self.flip_prob = flip_prob
        self.rotation_degrees = rotation_degrees

    def __call__(self, image, mask):
        # Random Horizontal Flip
        if random.random() < self.flip_prob:
            image = transforms.functional.hflip(image)
            mask = transforms.functional.hflip(mask)

        # Random Vertical Flip
        if random.random() < self.flip_prob:
            image = transforms.functional.vflip(image)
            mask = transforms.functional.vflip(mask)

        # Random Rotation
        angle = random.choice(self.rotation_degrees)
        image = transforms.functional.rotate(image, angle)
        mask = transforms.functional.rotate(mask, angle)

        return image, mask

# ------------------------------
# Custom Dataset Definition
# ------------------------------
class HAM10000Dataset(Dataset):
    """
    Custom Dataset for HAM10000 data that loads:
      - An image (resized to IMAGE_SIZE)
      - A corresponding segmentation mask (resized to IMAGE_SIZE)
      - A classification label from the dx_type column in metadata
    """
    def __init__(self, df, image_dirs, mask_dir, label_dict=None, transform=None, sync_transform=None, image_size=(256,256)):
        """
        Args:
            df: pandas DataFrame with columns 'image_id' and 'dx_type'.
            image_dirs: List of directories containing image files (with .jpg extension).
            mask_dir: Directory containing the segmentation masks (with _segmentation.png suffix).
            label_dict: Mapping from dx_type (string) to integer label. Example: {'nv': 0, 'mel': 1, ...}
            transform: Transformations to be applied to both image and mask (e.g. ToTensor).
            sync_transform: Synchronized transformations (e.g., flips and rotations) for both image and mask.
            image_size: Tuple (width, height) for resizing images and masks.
        """
        self.df = df
        self.image_dirs = image_dirs
        self.mask_dir = mask_dir
        self.label_dict = label_dict if label_dict is not None else {"histo": 0}
        self.transform = transform
        self.sync_transform = sync_transform
        self.image_size = image_size

        # Build list of indices that have both image and mask files.
        self.valid_indices = []
        for idx in range(len(self.df)):
            image_id = self.df.iloc[idx]["image_id"]
            if self.image_exists(image_id) and self.mask_exists(image_id):
                self.valid_indices.append(idx)

    def image_exists(self, image_id):
        for d in self.image_dirs:
            if os.path.exists(os.path.join(d, f"{image_id}.jpg")):
                return True
        return False

    def mask_exists(self, image_id):
        return os.path.exists(os.path.join(self.mask_dir, f"{image_id}_mask_mask.png"))

    def load_image(self, image_id):
        for d in self.image_dirs:
            path = os.path.join(d, f"{image_id}.jpg")
            if os.path.exists(path):
                return Image.open(path).convert("RGB")
        return None

    def load_mask(self, image_id):
        path = os.path.join(self.mask_dir, f"{image_id}_mask_mask.png")
        if os.path.exists(path):
            return Image.open(path).convert("L")
        return None

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # Use the valid index from our filtered list
        row_idx = self.valid_indices[idx]
        row = self.df.iloc[row_idx]
        image_id = row["image_id"]
        dx_type_str = row["dx_type"]
        label = self.label_dict[dx_type_str] if dx_type_str in self.label_dict else 0

        # Load image and mask
        image = self.load_image(image_id)
        mask = self.load_mask(image_id)

        # Resize both image and mask
        if self.image_size:
            image = image.resize(self.image_size)
            mask = mask.resize(self.image_size)

        # Apply synchronized transformations if provided
        if self.sync_transform:
            image, mask = self.sync_transform(image, mask)

        # Apply basic transformations
        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)
            mask = (mask > 0.5).float()  # Binarize mask
        else:
            image = transforms.ToTensor()(image)
            mask = transforms.ToTensor()(mask)
            mask = (mask > 0.5).float()

        return image, mask, label

# ------------------------------
# Load Metadata and Split Data
# ------------------------------
# Load metadata CSV
metadata = pd.read_csv(metadata_path)

# Define your label mapping (update according to your classes)
label_dict = {'nv': 0, 'mel': 1, 'bkl': 2, 'bcc': 3, 'akiec': 4, 'vasc': 5, 'df': 6}

# Split the metadata into training and validation sets (80/20 split)
train_df, val_df = train_test_split(metadata, test_size=0.2, random_state=42)

# ------------------------------
# Define Transformations
# ------------------------------
basic_transform = transforms.ToTensor()
sync_transform = SynchronizedTransform(flip_prob=0.5)

# ------------------------------
# Create Datasets and DataLoaders
# ------------------------------
train_dataset = HAM10000Dataset(
    df=train_df,
    image_dirs=image_dirs,
    mask_dir=mask_dir,
    label_dict=label_dict,
    transform=basic_transform,
    sync_transform=sync_transform,
    image_size=IMAGE_SIZE
)

val_dataset = HAM10000Dataset(
    df=val_df,
    image_dirs=image_dirs,
    mask_dir=mask_dir,
    label_dict=label_dict,
    transform=basic_transform,
    sync_transform=None,  # No augmentation for validation
    image_size=IMAGE_SIZE
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Loaded {len(train_dataset)} training samples and {len(val_dataset)} validation samples.")


Loaded 37548 training samples and 9387 validation samples.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reuse your existing ConvBlock, AttentionGate
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        return x

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super(AttentionGate, self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = F.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


class AttentionUNetMultiTask(nn.Module):
    """
    Attention U-Net with an additional classification head.
    """
    def __init__(self, img_ch=3, output_ch=1, num_classes=7):
        """
        Args:
            img_ch: Number of input channels (3 for RGB).
            output_ch: Number of segmentation channels (1 for binary).
            num_classes: Number of classification categories (e.g., 2, 3, etc.).
        """
        super(AttentionUNetMultiTask, self).__init__()

        # 1) ENCODER
        self.conv1 = ConvBlock(img_ch, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = ConvBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.conv3 = ConvBlock(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.conv4 = ConvBlock(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        self.conv5 = ConvBlock(512, 1024)

        # 2) DECODER (with Attention)
        self.att4 = AttentionGate(512, 512, 256)
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = ConvBlock(1024, 512)

        self.att3 = AttentionGate(256, 256, 128)
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(512, 256)

        self.att2 = AttentionGate(128, 128, 64)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(256, 128)

        self.att1 = AttentionGate(64, 64, 32)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(128, 64)

        self.final = nn.Conv2d(64, output_ch, kernel_size=1)

        # 3) CLASSIFICATION HEAD
        # We'll do a simple global average pooling on e5 => fc => classification logits
        self.global_pool = nn.AdaptiveAvgPool2d((1,1))
        self.classifier = nn.Linear(1024, num_classes)

    def forward(self, x):
        # ==================
        # ENCODER
        # ==================
        e1 = self.conv1(x)
        p1 = self.pool1(e1)

        e2 = self.conv2(p1)
        p2 = self.pool2(e2)

        e3 = self.conv3(p2)
        p3 = self.pool3(e3)

        e4 = self.conv4(p3)
        p4 = self.pool4(e4)

        e5 = self.conv5(p4)  # deepest encoder feature

        # ==================
        # DECODER (Segmentation)
        # ==================
        d4 = self.up4(e5)
        x4 = self.att4(g=d4, x=e4)
        d4 = torch.cat((x4, d4), dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        x3 = self.att3(g=d3, x=e3)
        d3 = torch.cat((x3, d3), dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        x2 = self.att2(g=d2, x=e2)
        d2 = torch.cat((x2, d2), dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        x1 = self.att1(g=d1, x=e1)
        d1 = torch.cat((x1, d1), dim=1)
        d1 = self.dec1(d1)

        seg_out = torch.sigmoid(self.final(d1))

        # ==================
        # CLASSIFICATION HEAD
        # ==================
        # Global average pool the deepest encoder feature e5 => shape [B, 1024, 1, 1]
        pooled = self.global_pool(e5)
        pooled = pooled.view(pooled.size(0), -1)  # shape [B, 1024]
        cls_out = self.classifier(pooled)         # shape [B, num_classes]

        return seg_out, cls_out


#Transfer learning from the previously trained attention unet

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Alias the old class name to your current class (only do this if you're sure it is safe)
AttentionUNet = AttentionUNetMultiTask  

# Now add this class to the safe globals
torch.serialization.add_safe_globals([AttentionUNet])

# Then load the checkpoint with weights_only=False
pretrained_path = r'C:\profolders\Collage stuff\Sem 4 project\DL\attention unet\attention_unet_model_final.pth'
pretrained_weights = torch.load(pretrained_path, map_location=device, weights_only=False)


In [ ]:
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AttentionUNetMultiTask(img_ch=3, output_ch=1, num_classes=len(label_dict)).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

NameError: name 'torch' is not defined

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Define Dice coefficient and loss, IoU and pixel accuracy as before
def dice_coefficient(pred, target, smooth=1e-5):
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return (2.0 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

class DiceLoss(nn.Module):
    def __init__(self):
        super(DiceLoss, self).__init__()
    def forward(self, pred, target):
        return 1.0 - dice_coefficient(pred, target)

def iou_score(pred, target, smooth=1e-5):
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + smooth) / (union + smooth)

def pixel_accuracy(pred, target):
    pred = pred.view(-1)
    target = target.view(-1)
    correct = (pred == target).sum().item()
    total = target.numel()
    return correct / total

# Classification loss: standard CrossEntropyLoss
cls_criterion = nn.CrossEntropyLoss()

# Combined training function that returns losses and metrics
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    running_loss = 0.0
    running_seg_loss = 0.0
    running_cls_loss = 0.0
    dice_scores = []
    iou_scores = []
    pixel_accuracies = []

    for images, masks, labels in dataloader:
        images = images.to(device)
        masks = masks.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        seg_out, cls_out = model(images)

        # SEGMENTATION LOSS
        seg_loss = DiceLoss()(seg_out, masks)

        # CLASSIFICATION LOSS
        cls_loss = cls_criterion(cls_out, labels)

        # TOTAL LOSS
        loss = seg_loss + cls_loss
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_seg_loss += seg_loss.item()
        running_cls_loss += cls_loss.item()

        # Binarize segmentation predictions for metrics calculation
        seg_preds = (seg_out > 0.5).float()
        dice_scores.append(dice_coefficient(seg_preds, masks).item())
        iou_scores.append(iou_score(seg_preds, masks).item())
        pixel_accuracies.append(pixel_accuracy(seg_preds, masks))

    epoch_loss = running_loss / len(dataloader)
    epoch_seg_loss = running_seg_loss / len(dataloader)
    epoch_cls_loss = running_cls_loss / len(dataloader)
    epoch_dice = sum(dice_scores) / len(dice_scores)
    epoch_iou = sum(iou_scores) / len(iou_scores)
    epoch_pixel_acc = sum(pixel_accuracies) / len(pixel_accuracies)

    return epoch_loss, epoch_seg_loss, epoch_cls_loss, epoch_dice, epoch_iou, epoch_pixel_acc

# Example training loop with saving losses/metrics for plotting:
num_epochs = 5

# Lists to store metrics for each epoch
train_loss_history      = []
train_seg_loss_history  = []
train_cls_loss_history  = []
train_dice_history      = []
train_iou_history       = []
train_pixel_acc_history = []

# Assuming you have already defined:
# - model (your multi-task AttentionUNetMultiTask)
# - train_loader (DataLoader for training data)
# - optimizer (e.g., optim.Adam(model.parameters(), lr=1e-4))
# - device (cuda or cpu)

for epoch in range(num_epochs):
    epoch_loss, epoch_seg_loss, epoch_cls_loss, epoch_dice, epoch_iou, epoch_pixel_acc = train_one_epoch(model, train_loader, optimizer, device)
    
    train_loss_history.append(epoch_loss)
    train_seg_loss_history.append(epoch_seg_loss)
    train_cls_loss_history.append(epoch_cls_loss)
    train_dice_history.append(epoch_dice)
    train_iou_history.append(epoch_iou)
    train_pixel_acc_history.append(epoch_pixel_acc)
    
    print(f"[Epoch {epoch+1}/{num_epochs}] Total Loss: {epoch_loss:.4f} | Seg Loss: {epoch_seg_loss:.4f} | Cls Loss: {epoch_cls_loss:.4f} | Dice: {epoch_dice:.4f} | IoU: {epoch_iou:.4f} | Pixel Acc: {epoch_pixel_acc:.4f}")

# Plotting the losses and metrics after training
epochs_range = range(1, num_epochs+1)

plt.figure(figsize=(12, 8))

# Total Loss
plt.subplot(2, 2, 1)
plt.plot(epochs_range, train_loss_history, label='Total Loss', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Total Loss over Epochs')
plt.legend()

# Segmentation and Classification Loss
plt.subplot(2, 2, 2)
plt.plot(epochs_range, train_seg_loss_history, label='Seg Loss', marker='o')
plt.plot(epochs_range, train_cls_loss_history, label='Cls Loss', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Segmentation and Classification Loss')
plt.legend()

# Dice Coefficient
plt.subplot(2, 2, 3)
plt.plot(epochs_range, train_dice_history, label='Dice Coefficient', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Dice Score')
plt.title('Dice Coefficient over Epochs')
plt.legend()

# IoU and Pixel Accuracy
plt.subplot(2, 2, 4)
plt.plot(epochs_range, train_iou_history, label='IoU', marker='o')
plt.plot(epochs_range, train_pixel_acc_history, label='Pixel Accuracy', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.title('IoU and Pixel Accuracy over Epochs')
plt.legend()

plt.tight_layout()
plt.savefig('training_metrics.png')
plt.show()


KeyboardInterrupt: 

In [ ]:
import os
import random
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ------------------------------
# Define Directories and File Paths
# ------------------------------
image_dirs = [
    r'C:\profolders\Collage stuff\Sem 4 project\DL\Datasets\smote_maximum_clarity'
]
mask_dir = r'C:\profolders\Collage stuff\Sem 4 project\DL\Datasets\smote_maximum_clarity_masks'
metadata_path = r'C:\profolders\Collage stuff\Sem 4 project\DL\Datasets\smote_maximum_clarity\smote_metadata.csv'
OUTPUT_OVERLAY_DIR = r'C:\profolders\Collage stuff\Sem 4 project\DL\smote_overlays'

# ------------------------------
# Constants
# ------------------------------
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 1  # Use 1 for overlay generation so we can process one image at a time

# ------------------------------
# Synchronized Transformations (Optional)
# ------------------------------
class SynchronizedTransform:
    def __init__(self, flip_prob=0.5, rotation_degrees=(0, 90, 180, 270)):
        self.flip_prob = flip_prob
        self.rotation_degrees = rotation_degrees

    def __call__(self, image, mask):
        if random.random() < self.flip_prob:
            image = transforms.functional.hflip(image)
            mask = transforms.functional.hflip(mask)
        if random.random() < self.flip_prob:
            image = transforms.functional.vflip(image)
            mask = transforms.functional.vflip(mask)
        angle = random.choice(self.rotation_degrees)
        image = transforms.functional.rotate(image, angle)
        mask = transforms.functional.rotate(mask, angle)
        return image, mask

# ------------------------------
# Custom Dataset for SMOTE Data
# ------------------------------
class SmoteDataset(Dataset):
    """
    Loads data from the SMOTE dataset.
    Assumes:
      - Image file: "{image_id}.jpg" from one of the directories in image_dirs.
      - Mask file: "{image_id}_mask_mask.png" from mask_dir.
      - Metadata CSV contains at least the column "image_id".
    """
    def __init__(self, metadata_path, image_dirs, mask_dir, transform=None, sync_transform=None, image_size=(256,256)):
        self.df = pd.read_csv(metadata_path)
        self.image_dirs = image_dirs
        self.mask_dir = mask_dir
        self.transform = transform
        self.sync_transform = sync_transform
        self.image_size = image_size

        # Filter for images that exist along with their corresponding masks
        self.valid_indices = []
        for idx in range(len(self.df)):
            image_id = self.df.iloc[idx]["image_id"]
            if self.image_exists(image_id) and self.mask_exists(image_id):
                self.valid_indices.append(idx)

    def image_exists(self, image_id):
        for d in self.image_dirs:
            if os.path.exists(os.path.join(d, f"{image_id}.jpg")):
                return True
        return False

    def mask_exists(self, image_id):
        mask_path = os.path.join(self.mask_dir, f"{image_id}_mask_mask.png")
        return os.path.exists(mask_path)

    def load_image(self, image_id):
        for d in self.image_dirs:
            path = os.path.join(d, f"{image_id}.jpg")
            if os.path.exists(path):
                return Image.open(path).convert("RGB")
        return None

    def load_mask(self, image_id):
        mask_path = os.path.join(self.mask_dir, f"{image_id}_mask_mask.png")
        if os.path.exists(mask_path):
            return Image.open(mask_path).convert("L")
        return None

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        row_idx = self.valid_indices[idx]
        row = self.df.iloc[row_idx]
        image_id = row["image_id"]

        image = self.load_image(image_id)
        mask = self.load_mask(image_id)

        # Resize
        if self.image_size:
            image = image.resize(self.image_size)
            mask = mask.resize(self.image_size)

        # Apply synchronized transforms (if any)
        if self.sync_transform:
            image, mask = self.sync_transform(image, mask)

        # Apply basic transform: convert to tensor
        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)
            mask = (mask > 0.5).float()  # Binarize mask
        else:
            image = transforms.ToTensor()(image)
            mask = transforms.ToTensor()(mask)
            mask = (mask > 0.5).float()

        return image, mask, image_id  # returning image_id for naming later

# ------------------------------
# Create Dataset and DataLoader
# ------------------------------
basic_transform = transforms.ToTensor()
sync_transform = SynchronizedTransform(flip_prob=0.5)

dataset = SmoteDataset(
    metadata_path=metadata_path,
    image_dirs=image_dirs,
    mask_dir=mask_dir,
    transform=basic_transform,
    sync_transform=sync_transform,
    image_size=IMAGE_SIZE
)

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

# ------------------------------
# Segmentation Overlay Visualization Function
# ------------------------------
def visualize_segmentation_overlay(image_tensor, gt_mask_tensor, pred_mask_tensor, alpha=0.4):
    """
    Overlays ground truth (green) and predicted (red) binary masks on the original image.
    Overlap is shown in purple.
    
    Returns:
        fig: The matplotlib Figure object.
    """
    # Convert image tensor [C,H,W] to numpy image [H,W,C]
    image_np = image_tensor.permute(1, 2, 0).cpu().numpy()
    image_np = np.clip(image_np, 0, 1)
    
    gt_mask_np   = gt_mask_tensor.squeeze().cpu().numpy()  # shape [H,W]
    pred_mask_np = pred_mask_tensor.squeeze().cpu().numpy()  # shape [H,W]

    # Define colors
    color_gt      = np.array([0, 1, 0])   # green
    color_pred    = np.array([1, 0, 0])   # red
    color_overlap = np.array([1, 0, 1])   # purple

    overlay = image_np.copy()

    overlap_mask   = (gt_mask_np == 1) & (pred_mask_np == 1)
    just_gt_mask   = (gt_mask_np == 1) & (pred_mask_np == 0)
    just_pred_mask = (gt_mask_np == 0) & (pred_mask_np == 1)

    overlay[just_gt_mask]   = (1 - alpha) * overlay[just_gt_mask]   + alpha * color_gt
    overlay[just_pred_mask] = (1 - alpha) * overlay[just_pred_mask] + alpha * color_pred
    overlay[overlap_mask]   = (1 - alpha) * overlay[overlap_mask]   + alpha * color_overlap

    fig, ax = plt.subplots(figsize=(6,6))
    ax.imshow(overlay)
    ax.set_title("Overlay: GT (Green), Pred (Red), Overlap (Purple)")
    ax.axis('off')

    legend_patches = [
        Patch(color=(0,1,0), label='Ground Truth'),
        Patch(color=(1,0,0), label='Prediction'),
        Patch(color=(1,0,1), label='Overlap')
    ]
    ax.legend(handles=legend_patches, loc='upper right')
    return fig

# ------------------------------
# Model Loading and Inference
# ------------------------------
# Here you should load your pre-trained model.
# For demonstration, we'll assume that your model is defined as AttentionUNetMultiTask
# and that the variable 'model' is already loaded.
#
# For example:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = AttentionUNetMultiTask(img_ch=3, output_ch=1, num_classes=7).to(device)
#
# If you have a segmentation-only model, adjust accordingly.
# For this demo, we'll assume 'model' is defined.
#
# --- BEGIN: Load your model here ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Replace the following line with your actual model definition/loading.
model = torch.load(r'C:\profolders\Collage stuff\Sem 4 project\DL\attention_unet_model_final.pth', map_location=device)
model.eval()
# --- END: Load your model ---

# ------------------------------
# Generate and Save Overlays Automatically
# ------------------------------
def generate_and_save_overlays(model, dataloader, device, output_dir, max_images=10):
    os.makedirs(output_dir, exist_ok=True)
    count = 0

    with torch.no_grad():
        for batch in dataloader:
            if count >= max_images:
                break

            # Since batch_size=1, unpack directly
            image, mask, image_id = batch  # image_id is a list with one element
            image = image.to(device)
            mask = mask.to(device)

            seg_out, _ = model(image)  # we only need segmentation output here

            # Binarize prediction (assuming single-channel binary segmentation)
            seg_pred = (seg_out > 0.5).float()

            fig = visualize_segmentation_overlay(
                image_tensor=image[0].cpu(),
                gt_mask_tensor=mask[0].cpu(),
                pred_mask_tensor=seg_pred[0].cpu(),
                alpha=0.4
            )

            # Use the image_id for naming output file
            output_path = os.path.join(output_dir, f"{image_id[0]}_overlay.png")
            fig.savefig(output_path, bbox_inches='tight')
            plt.close(fig)
            print(f"Saved overlay to {output_path}")
            count += 1

# Run the overlay generation on your dataset
generate_and_save_overlays(model, loader, device, OUTPUT_OVERLAY_DIR, max_images=10)

In [ ]:
import torch
from sklearn.metrics import confusion_matrix
import seaborn as sns

def evaluate_classification(model, dataloader, device, num_classes=7):
    """
    Runs inference on the dataloader, collects classification predictions and labels.
    Returns lists of ground truth labels and predicted labels.
    """
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, masks, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            seg_out, cls_out = model(images)

            # Get predicted class from logits
            preds = torch.argmax(cls_out, dim=1)

            all_labels.extend(labels.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())

    return all_labels, all_preds


In [ ]:
def plot_confusion_matrix(true_labels, pred_labels, class_names):
    """
    Plots a confusion matrix given true and predicted labels.
    """
    cm = confusion_matrix(true_labels, pred_labels)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("Ground Truth")
    plt.title("Confusion Matrix")
    plt.show()
    plt.cl

In [ ]:
# Suppose you have a validation dataloader called `val_loader`,
# a model on `device`, and a list of class names:
class_names = ["nv","mel","bkl","bcc","akiec","vasc","df"]

# Evaluate classification predictions
true_labels, pred_labels = evaluate_classification(model, val_loader, device, num_classes=7)

# Plot the confusion matrix
plot_confusion_matrix(true_labels, pred_labels, class_names)
